<img src="https://github.com/ArchiColab/sitex/blob/main/images/banner.png?raw=1" width="1000"/>

# Add Missing Streets to the OSM Network

OpenStreetMap is often behind on new roads. This notebook takes streets you have drawn
yourself and joins them to the OSM street network, then saves the result as a new street
network GeoPackage that the other notebooks can read.

**The question:** a line drawn on top of a map only *looks* connected. For Space Syntax or
an isochrone, a street exists only if it shares a junction with the rest of the network.
So the notebook does what is tedious by hand in QGIS:

1. **Snap:** each end of a new street is moved onto the network: to an existing junction if
   one is close, otherwise onto the nearest street.
2. **Split:** the existing street is cut where the new street joins it, and a new junction
   is added there.
3. **Save:** nodes and edges are written in the same layout as the OSM file
   (`{slug}_{network_type}_updated.gpkg`, layers `nodes` and `edges`).

All the calculations live in `sitex.network.street_edits`; this notebook only sets the
parameters, calls the functions, and shows the result.

**Library:** [`sitex`](https://github.com/ArchiColab/sitex)

**Requires (run first):**
- `00-Data-Acquisition.ipynb` — Section 9, OSM street networks (`osm/{slug}_drive.gpkg`, …)
- Your own sketch of the missing streets, as lines (see below)

**Drawing the missing streets in QGIS** (no snapping needed, the notebook does it):

1. Load the `edges` layer of `data/osm/{slug}_drive.gpkg` and a satellite basemap.
2. *Layer ▸ Create Layer ▸ New Temporary Scratch Layer*, geometry type **LineString**.
3. *Add Line Feature*: draw the centreline of each missing street. Start and end each line
   close to the street it joins (within a few metres).
4. Optional: add text columns `name` and `highway` (e.g. `residential`, `tertiary`).
5. Right-click the layer ▸ *Export ▸ Save Features As…* ▸ **GeoJSON**, and save it as
   `data/osm/new_street.geojson`.

---
## 1. Install & import

In [1]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    from google.colab import drive
    drive.mount("/content/gdrive")

    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")

    if not (DATA_DIR / "osm").is_dir():
        raise FileNotFoundError(
            f"{DATA_DIR} not found (or incomplete) in your Google Drive. "
            "Before the workshop: download the workshop data zip and unzip it into "
            "My Drive/Colab_Outputs -- see the workshop setup instructions."
        )
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import folium
import geopandas as gpd
import pandas as pd

from sitex.core.config import CityConfig
from sitex.network import street_edits as se

Running in: Local environment


## 2. Site configuration & parameters

`slug` must be the same as in the data acquisition (default `pleiku`).

| Parameter | Meaning |
|---|---|
| `NEW_STREETS_FILE` | Your sketch: GeoJSON or GeoPackage with lines. Several streets and MultiLineStrings are fine. |
| `NETWORK_TYPES` | Which OSM networks get the new streets. A new road is usually both drivable and walkable. |
| `SNAP_TOLERANCE_M` | How far (m) a sketched end may be moved to reach the network. An end with nothing in range stays a dead end. |
| `CONNECT_CROSSINGS` | `True`: where a new street crosses another street, they are joined as a junction. `False`: crossings are bridges or underpasses; only the two ends connect. |
| `HIGHWAY`, `NAME` | OSM road class and street name for the new streets, unless the sketch has its own `highway` / `name` columns. |

In [2]:
_t0 = time.time()
print("### 2. Site configuration & parameters ###", flush=True)

city = CityConfig(
    place_name="Phường Pleiku, Gia Lai, Vietnam",
    local_lat=13.9833,
    local_lon=108.0000,
    slug="pleiku",
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)
OSM_DIR = city.data_dir / "osm"

NEW_STREETS_FILE = OSM_DIR / "new_street.geojson"
NETWORK_TYPES = ["drive", "walk", "drive_service"]
SNAP_TOLERANCE_M = 5.0
CONNECT_CROSSINGS = True
HIGHWAY = "residential"
NAME = ""

new_streets = gpd.read_file(NEW_STREETS_FILE).to_crs(epsg=city.local_epsg)
sketch = se.sketch_to_lines(new_streets)
print(f"Loaded {len(sketch)} new street(s), {sketch.length.sum():,.0f} m in total  (EPSG:{city.local_epsg})")

print(f"--- 2. done in {time.time() - _t0:.2f}s ---\n")

### 2. Site configuration & parameters ###


Loaded 1 new street(s), 368 m in total  (EPSG:32649)
--- 2. done in 0.06s ---



C:\Users\Maddie\anaconda3\envs\gis\lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


---
## 3. Snap the new streets and join them to the network

Each network is read from the original OSM file, so running this again never adds a street
twice. The table lists every sketched street end: what it was snapped to and how far it
was moved. **Read it before you go on:**

- `existing junction` or `along an existing street`: connected.
- `nothing (dead end)`: no street within `SNAP_TOLERANCE_M`. Right for a cul-de-sac, wrong
  if the street should connect. Then draw the end closer, or raise the tolerance.
- A large `moved_m` on a short street changes its shape. Redraw it rather than accept it.

In [3]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 3. Snap the new streets and join them to the network ###", flush=True)

updated, reports = {}, []
for network_type in NETWORK_TYPES:
    nodes, edges = se.load_network(OSM_DIR / f"{city.slug}_{network_type}.gpkg")
    nodes = nodes.to_crs(epsg=city.local_epsg)
    edges = edges.to_crs(epsg=city.local_epsg)

    nodes_new, edges_new, report = se.add_streets_to_network(
        nodes, edges, new_streets,
        snap_tolerance=SNAP_TOLERANCE_M, connect_crossings=CONNECT_CROSSINGS,
        highway=HIGHWAY, name=NAME,
    )
    updated[network_type] = (nodes_new, edges_new)
    reports.append(report.assign(network=network_type))

    added = edges_new[edges_new["source"] == "added"]
    connected = se.in_main_network(edges_new)[added.index]
    print(
        f"{network_type:14s} {len(edges):,} -> {len(edges_new):,} edges, "
        f"{len(nodes):,} -> {len(nodes_new):,} nodes  |  "
        f"{len(added)} new edge(s), {added.length.sum():,.0f} m, "
        f"{connected.sum()} of {len(added)} connected to the main network"
    )
    if not connected.all():
        print(f"  ⚠️ {(~connected).sum()} new edge(s) are NOT connected. Check the table below.")

print(f"--- 3. done in {time.time() - _t0:.2f}s ---\n")
pd.concat(reports).set_index(["network", "street", "end"])

### 3. Snap the new streets and join them to the network ###


drive          2,210 -> 2,211 edges, 1,748 -> 1,748 nodes  |  1 new edge(s), 364 m, 1 of 1 connected to the main network


walk           3,344 -> 3,345 edges, 2,673 -> 2,673 nodes  |  1 new edge(s), 364 m, 1 of 1 connected to the main network


drive_service  3,272 -> 3,273 edges, 2,614 -> 2,614 nodes  |  1 new edge(s), 364 m, 1 of 1 connected to the main network
--- 3. done in 0.42s ---



snapped_to  moved_m
network       street end                              
drive         0      start  existing junction      4.4
                     end    existing junction      0.0
walk          0      start  existing junction      4.4
                     end    existing junction      0.0
drive_service 0      start  existing junction      4.4
                     end    existing junction      0.0

## 4. Check on the map

Grey: the OSM network. Red: the streets as they were added. Blue dashed: your sketch as you
drew it. Black dots: junctions that did not exist before. Zoom in on each end of a red line
and check that it meets the grey network where you meant it to.

In [4]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 4. Check on the map ###", flush=True)

nodes_new, edges_new = updated[NETWORK_TYPES[0]]
is_added = edges_new["source"] == "added"
show_cols = ["highway", "name", "length", "source", "geometry"]

m = folium.Map(tiles="cartodbpositron", control_scale=True)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery", name="Satellite", overlay=False,
).add_to(m)

folium.GeoJson(
    edges_new.loc[~is_added, show_cols].to_crs(epsg=4326), name=f"OSM network ({NETWORK_TYPES[0]})",
    style_function=lambda f: {"color": "#777777", "weight": 2},
    tooltip=folium.GeoJsonTooltip(["highway", "name"]),
).add_to(m)
folium.GeoJson(
    sketch[["geometry"]].to_crs(epsg=4326), name="Sketch, as drawn",
    style_function=lambda f: {"color": "#1f78b4", "weight": 2, "dashArray": "6 6"},
).add_to(m)
added_wgs = edges_new.loc[is_added, show_cols].to_crs(epsg=4326)
folium.GeoJson(
    added_wgs, name="Added streets",
    style_function=lambda f: {"color": "#e31a1c", "weight": 5},
    tooltip=folium.GeoJsonTooltip(["highway", "name", "length"]),
).add_to(m)
for pt in nodes_new.loc[nodes_new["source"] == "added"].to_crs(epsg=4326).geometry:
    folium.CircleMarker((pt.y, pt.x), radius=5, color="black", fill=True, fill_opacity=1,
                        tooltip="New junction").add_to(m)

west, south, east, north = added_wgs.total_bounds
m.fit_bounds([(south, west), (north, east)], padding=(80, 80))
folium.LayerControl(collapsed=False).add_to(m)

print(f"--- 4. done in {time.time() - _t0:.2f}s ---\n")
m

### 4. Check on the map ###


--- 4. done in 0.39s ---



---
## 5. Save the updated network

One GeoPackage per network type, next to the original: `osm/{slug}_{network_type}_updated.gpkg`.
The original OSM files are not changed. In QGIS, style the `edges` layer by the `source`
column (`osm` / `added`) to see the new streets.

In [5]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }

_t0 = time.time()
print("### 5. Save the updated network ###", flush=True)

for network_type, (nodes_new, edges_new) in updated.items():
    path = se.save_network(nodes_new, edges_new, OSM_DIR / f"{city.slug}_{network_type}_updated.gpkg")
    print(f"{network_type:14s} {len(edges_new):,} edges  -> {path}")

print(f"--- 5. done in {time.time() - _t0:.2f}s ---\n")

### 5. Save the updated network ###


drive          2,211 edges  -> ..\data\osm\pleiku_drive_updated.gpkg


C:\Users\Maddie\anaconda3\envs\gis\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Cannot find tms_NZTM2000.json (GDAL_DATA is not defined)
  ogr_write(


walk           3,345 edges  -> ..\data\osm\pleiku_walk_updated.gpkg


drive_service  3,273 edges  -> ..\data\osm\pleiku_drive_service_updated.gpkg
--- 5. done in 0.41s ---



---
## Use the updated network

The other notebooks read `osm/{slug}_drive.gpkg` or `osm/{slug}_walk.gpkg`. To analyse the
updated network, change that one file name and run the notebook again:

| Notebook | Line to change |
|---|---|
| [`03-NA01-SpaceSyntax.ipynb`](03-NA01-SpaceSyntax.ipynb), Section 2 | `STREETS_FILE = city.data_dir / "osm" / f"{city.slug}_drive_updated.gpkg"` |
| [`03-NA01-SpaceSyntax_FieldTripIntro.ipynb`](03-NA01-SpaceSyntax_FieldTripIntro.ipynb), Section 2 | `STREETS_FILE = city.data_dir / "osm" / f"{city.slug}_walk_updated.gpkg"` |
| Isochrone and accessibility notebooks (`03-NA03` to `03-NA06`) | the path ending in `_walk.gpkg` → `_walk_updated.gpkg` |

## Notes

- **Existing road or design proposal?** This notebook is for streets that are already built
  but missing in OSM: they become part of the baseline. To test a street you *propose*,
  keep the baseline as it is and use the scenario test in `03-NA01-SpaceSyntax.ipynb`,
  Section 10. If the same `new_street.geojson` is used for both, the scenario test compares
  the street with itself.
- **The snap tolerance is a method choice.** An end is moved to a junction first, and only
  onto the middle of a street if no junction is within the tolerance. With a large
  tolerance a street that should join mid-block jumps to the corner instead.
- **A bridge is not a junction.** With `CONNECT_CROSSINGS = True` every crossing becomes a
  junction. For a flyover or underpass, set it to `False`.
- **New objects have negative ids.** New junctions and streets get ids −1, −2, … so they
  can never be confused with real OSM ids. Split OSM streets keep their `osmid` and name.
- **Better still: add the road to OpenStreetMap.** Then everyone gets it, and the next
  download in `00-Data-Acquisition.ipynb` includes it without this step.